<a href="https://colab.research.google.com/github/simjonghyeon04/-/blob/main/%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC%20%EA%B3%BC%EC%A0%9C%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn python-multipart httpx beautifulsoup4 pandas matplotlib seaborn gradio pyngrok nest-asyncio

In [ ]:
import asyncio
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import httpx
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException
import gradio as gr
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from collections import Counter
import re
import os

# 1. 환경 설정
nest_asyncio.apply()
app = FastAPI(title="Quotes Management API")
DB_NAME = "assignment_reset.db"  # DB 파일 이름

# [수정] 실행할 때마다 DB를 초기화(삭제 후 재생성)하는 함수
def init_db():
    # 만약 기존 DB 파일이 있다면 삭제 (초기화 핵심)
    if os.path.exists(DB_NAME):
        os.remove(DB_NAME)
        print(f"♻️ 기존 데이터베이스({DB_NAME})를 초기화했습니다.")

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute('''CREATE TABLE IF NOT EXISTS quotes
                          (id INTEGER PRIMARY KEY AUTOINCREMENT, quote TEXT, author TEXT, tag TEXT)''')

        # 무조건 새로 20개 수집
        print("🌐 웹사이트에서 신선한 격언 20개를 수집하고 있습니다...")
        collected = []
        page = 1
        with httpx.Client() as client:
            while len(collected) < 20:
                res = client.get(f"https://quotes.toscrape.com/page/{page}/")
                if res.status_code != 200: break
                soup = BeautifulSoup(res.text, 'html.parser')
                items = soup.find_all('div', class_='quote')
                if not items: break
                for item in items:
                    if len(collected) >= 20: break
                    text = item.find('span', class_='text').text
                    author = item.find('small', class_='author').text
                    tag = item.find('a', class_='tag').text if item.find('a', class_='tag') else "Life"
                    collected.append((text, author, tag))
                page += 1
        cursor.executemany("INSERT INTO quotes (quote, author, tag) VALUES (?, ?, ?)", collected)
        conn.commit()
    print("✅ 초기화 및 데이터 수집 완료.")

# 2. 기능 함수
def load_data():
    try:
        with sqlite3.connect(DB_NAME) as conn:
            return pd.read_sql("SELECT * FROM quotes ORDER BY id ASC", conn)
    except:
        return pd.DataFrame(columns=["id", "quote", "author", "tag"])

def add_quote(q, a, t):
    if not q or not a: return "❌ 내용과 저자를 입력하세요."
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("INSERT INTO quotes (quote, author, tag) VALUES (?, ?, ?)", (q, a, t))
    return "✅ 추가되었습니다."

def delete_quote(quote_id):
    if quote_id is None: return "❌ ID를 입력하세요."
    with sqlite3.connect(DB_NAME) as conn:
        cur = conn.cursor()
        cur.execute("DELETE FROM quotes WHERE id = ?", (quote_id,))
    return f"✅ ID {int(quote_id)} 삭제 완료."

def get_random_quote():
    df = load_data()
    if df.empty: return "데이터가 없습니다."
    row = df.sample(1).iloc[0]
    return f"“{row['quote']}”\n\n───\n👤 저자: {row['author']}\n🏷️ 태그: {row['tag']}"

def get_plots():
    df = load_data()
    if df.empty: return None, None
    fig1, ax1 = plt.subplots(figsize=(10, 4))
    words = re.findall(r'\w+', " ".join(df['quote'].str.lower()))
    counts = Counter([w for w in words if len(w) > 4]).most_common(10)
    sns.barplot(x=[c[1] for c in counts], y=[c[0] for c in counts], palette="flare", ax=ax1)
    ax1.set_title("Top 10 Words")

    fig2, ax2 = plt.subplots(figsize=(10, 4))
    df['author'].value_counts().head(10).plot.pie(autopct='%1.1f%%', ax=ax2, cmap='Set3')
    ax2.set_ylabel('')
    ax2.set_title("Author Distribution")
    plt.tight_layout()
    return fig1, fig2

# 3. FastAPI 설정
@app.get("/api/quotes")
async def api_get_all():
    return load_data().to_dict(orient="records")

# 4. Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# 🏛️ 명언 관리 시스템 (자동 리셋 버전)")

    with gr.Tab("📋 목록 & 삭제"):
        with gr.Row():
            btn_refresh = gr.Button("🔄 새로고침")
            del_id = gr.Number(label="삭제할 ID", precision=0)
            btn_del = gr.Button("🗑️ 삭제", variant="stop")
        data_grid = gr.Dataframe(value=load_data, interactive=False)

        btn_refresh.click(load_data, outputs=data_grid)
        btn_del.click(delete_quote, inputs=del_id).then(load_data, outputs=data_grid)

    with gr.Tab("📊 통계"):
        btn_visual = gr.Button("📊 분석")
        with gr.Row():
            out_plot1 = gr.Plot()
            out_plot2 = gr.Plot()
        btn_visual.click(get_plots, outputs=[out_plot1, out_plot2])

    with gr.Tab("🎲 추천 & 추가"):
        with gr.Row():
            with gr.Column():
                random_display = gr.Textbox(label="오늘의 격언", lines=10)
                btn_rand = gr.Button("🎲 뽑기")
            with gr.Column():
                txt_q = gr.Textbox(label="내용")
                txt_a = gr.Textbox(label="저자")
                txt_t = gr.Textbox(label="태그")
                btn_insert = gr.Button("💾 저장")
                msg_area = gr.Markdown()

        btn_rand.click(get_random_quote, outputs=random_display)
        btn_insert.click(add_quote, [txt_q, txt_a, txt_t], outputs=msg_area).then(load_data, outputs=data_grid)

# 5. 실행
async def main():
    init_db() # 여기서 기존 파일을 지우고 새로 시작합니다.

    NGROK_TOKEN = "3DNRU3mtymTM6aQiSHAvGuZShFu_5GpJRtjsk3jytkqDtoyHm"
    ngrok.set_auth_token(NGROK_TOKEN)
    try: ngrok.kill()
    except: pass

    combined_app = gr.mount_gradio_app(app, demo, path="/ui")
    public_url = ngrok.connect(8000)

    print(f"\n" + "="*60)
    print(f"🔗 [API Swagger] : {public_url.public_url}/docs")
    print(f"🔗 [Gradio UI]   : {public_url.public_url}/ui")
    print("="*60 + "\n")

    config = uvicorn.Config(combined_app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

if __name__ == "__main__":
    await main()

♻️ 기존 데이터베이스(assignment_reset.db)를 초기화했습니다.
🌐 웹사이트에서 신선한 격언 20개를 수집하고 있습니다...
✅ 초기화 및 데이터 수집 완료.
new /ui

🔗 [API Swagger] : https://clamp-buckwheat-sardine.ngrok-free.dev/docs
🔗 [Gradio UI]   : https://clamp-buckwheat-sardine.ngrok-free.dev/ui



INFO:     Started server process [4162]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui HTTP/1.1" 307 Temporary Redirect
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/ HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/theme.css?v=2fa39f6df94156f7ef9d84370dbb2657d672827c75f082ac995879c32290c5ca HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /manifest.json HTTP/1.1" 404 Not Found
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "POST /ui/gradio_api/queue/join HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/gradio_api/queue/data?session_hash=d0ixhfzbzw HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "POST /ui/gradio_api/queue/join HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/gradio_api/queue/data

/tmp/ipykernel_4162/438350558.py:89: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=[c[1] for c in counts], y=[c[0] for c in counts], palette="flare", ax=ax1)


INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/gradio_api/queue/data?session_hash=d0ixhfzbzw HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "POST /ui/gradio_api/queue/join HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/gradio_api/queue/data?session_hash=d0ixhfzbzw HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui HTTP/1.1" 307 Temporary Redirect
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/ HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /manifest.json HTTP/1.1" 404 Not Found
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /ui/theme.css?v=2fa39f6df94156f7ef9d84370dbb2657d672827c75f082ac995879c32290c5ca HTTP/1.1" 200 OK
INFO:     2001:2d8:7190:564a:df5:ca8f:616c:90d7:0 - "GET /m

/tmp/ipykernel_4162/438350558.py:89: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=[c[1] for c in counts], y=[c[0] for c in counts], palette="flare", ax=ax1)
